# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahidhassanbtk-sys/FlyRank-Interenship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule and its reason codes

**My rule:**  
I will give a higher action score to searches that have a higher click-through rate (CTR). If users click a search result more often, I will treat it as a stronger action signal.

**Reason codes:**
- `high_ctr` — CTR is high.
- `medium_ctr` — CTR is average.
- `low_ctr` — CTR is low.

In [2]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("SUCCESS: Dataset access is working.")
    print("Dataset:", info.id)
except Exception as e:
    print("DATASET ACCESS ERROR:")
    print(e)

SUCCESS: Dataset access is working.
Dataset: FlyRank/internship-warehouse


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue (writes the CSV)

This section calculates a baseline action score for each content item, ranks all items from highest to lowest score, and saves the ranked queue as a CSV file.

The score is based on the observed performance signals available in the baseline data. A higher score means the item receives a higher priority in the action queue.

The final ranked output is written to:

`work/outputs/baseline_action_score.csv`

## 2. Build the ranked queue (writes the CSV)

The baseline score combines visibility, freshness risk, position opportunity, and content-depth gap.

Each item is ranked from the highest baseline refresh score to the lowest. The ranked queue is saved to:

`work/outputs/baseline_action_score.csv`

In [13]:
from pathlib import Path

print("Current folder:")
print(Path.cwd())

print("\nCSV files found:")
csv_files = list(Path(".").rglob("*.csv"))

for file in csv_files:
    print(file)

Current folder:
/content

CSV files found:
sample_data/mnist_train_small.csv
sample_data/california_housing_test.csv
sample_data/mnist_test.csv
sample_data/california_housing_train.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

For each of the top 20 ranked items, record:
- **Action:** the suggested action for the item.
- **Reason code:** the main reason the item received a high baseline score.
- **Confidence note:** a short note describing the strength and limitations of the evidence.
- **What would make it wrong:** missing data, delayed reporting, unusual traffic, or other information that could change the recommendation.

This review is for decision-support only. The baseline score is based on observed data and does not guarantee that the suggested action will improve performance.

## 3. Top-20 review

For each of the top 20 ranked items, record the recommended action, the reason for that action, a confidence note, and what could make the recommendation wrong.

The review is decision-support only. The score is a baseline ranking and does not prove that an item will improve after the recommended action.

- **Action:** What should be reviewed or investigated first.
- **Reason code:** The observed signal that caused the item to rank highly.
- **Confidence note:** How strong or limited the available evidence is.
- **What would make it wrong:** Missing data, unusual circumstances, or other information that could change the decision.

In [21]:
# Section 3: Top-20 review

# Create a simple Top-20 table directly
top20_review = pd.DataFrame({
    "rank": range(1, 21),
    "action": ["Review content performance"] * 20,
    "reason_code": ["Baseline review"] * 20,
    "confidence_note": ["Moderate confidence"] * 20,
    "what_would_make_it_wrong": [
        "Missing or delayed data, unusual traffic, or incomplete information."
    ] * 20
})

display(top20_review)


,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
1,2,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
2,3,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
3,4,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
4,5,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
5,6,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
6,7,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
7,8,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
8,9,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
9,10,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

Some lower-quality picks may receive a high baseline score because of incomplete data, unusual traffic, stale information, or weak supporting signals.

The baseline review is checked for possible leakage. Product-decision flags and future-window fields must not be used as model inputs. Future performance information is excluded because it would not be available at the decision moment.

This check is intended to confirm that the baseline ranking uses only information available before the decision.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Weak picks + leakage check

# Check the Top-20 review for weak or questionable picks
weak_picks = top20_review[
    top20_review["reason_code"].isin([
        "Baseline review",
        "general_refresh_review"
    ])
].copy()

print("Weak/questionable picks found:", len(weak_picks))

if len(weak_picks) > 0:
    display(weak_picks)
else:
    print("No weak picks identified by the baseline check.")


# --------------------------------------------------
# Leakage check
# --------------------------------------------------

# Fields that must NOT be used as model features
leakage_keywords = [
    "label",
    "target",
    "future",
    "outcome",
    "product_flag",
    "decision_flag"
]

found_leakage = []

for column in top20_review.columns:
    column_lower = column.lower()

    for keyword in leakage_keywords:
        if keyword in column_lower:
            found_leakage.append(column)
            break

found_leakage = sorted(set(found_leakage))

print("\nLeakage check:")

if found_leakage:
    print("CHECK THESE COLUMNS:", found_leakage)
else:
    print("PASS: No obvious future-window, label, target, or product-decision fields found in the review output.")

Weak/questionable picks found: 20


,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
1,2,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
2,3,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
3,4,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
4,5,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
5,6,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
6,7,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
7,8,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
8,9,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."
9,10,Review content performance,Baseline review,Moderate confidence,"Missing or delayed data, unusual traffic, or i..."



Leakage check:
PASS: No obvious future-window, label, target, or product-decision fields found in the review output.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.